In [2]:
pip install gymnasium stable-baselines3 pandas numpy scikit-learn

  Using cached gymnasium-1.3.0-py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/953.9 kB ? eta -:--:--
   -------------------------------- ------- 786.4/953.9 kB 5.2 MB/s eta 0:00:01
   ---------------------------------------- 953.9/953.9 kB 4.1 MB/s  0:00:00
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   --------------- ------------------------ 3.9/10.0 MB 19.9 MB/s eta 0:00:01
   -------------------------------------- - 9.7/10.0 MB 22.9 MB/s eta 0:00:01
   ---------------------------------------- 10.0/10.0 MB 16.4 MB/s  0:00:00
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   --------------- ------------------------ 3.1/8.3 MB 13.9 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 19.2 MB/s  0:00:00
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   ---------- ----------------------------- 10.2/37.3 MB 49.2 MB/s eta 0:00:01
   ---------------------- ----------

In [5]:
pip install stable-baselines3[extra]

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------------------- ---- 1.6/1.8 MB 7.0 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 7.3 MB/s  0:00:00
   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ---------------------------------------  5.5/5.5 MB 26.4 MB/s eta 0:00:01
   ---------------------------------------- 5.5/5.5 MB 22.1 MB/s  0:00:00
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ---------------------------------------  5.2/5.3 MB 26.8 MB/s eta 0:00:01
   ---------------------------------------- 5.3/5.3 MB 23.7 MB/s  0:00:00
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------- ----------------- 6.0/10.6 MB 29.3 MB/s eta 0:00:01
   ---------------------------------------- 10.6/10.6 MB 27.7 MB/s  0:00:00

   ----------------------------------------  0/1

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
# Cần cài đặt: pip install gymnasium stable-baselines3 pandas numpy scikit-learn
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
from stable_baselines3 import DQN
from sklearn.preprocessing import StandardScaler
import os
import warnings
warnings.filterwarnings("ignore")

CSV_FILE = "focus_dataset.csv"

# Danh sách các cột đặc trưng sẽ đưa vào mạng Neural (State)
FEATURE_COLS = [
    'head_pitch', 'head_yaw', 'head_roll', 
    'ear_score', 'mar_score', 'brow_dist', 
    'person_detected', 'phone_count', 'consecutive_frames'
]



Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
class FocusManagementEnv(gym.Env):
    """
    Custom Environment mô phỏng môi trường làm việc dựa trên dữ liệu tĩnh từ CSV.
    DQN Agent sẽ tương tác với môi trường này theo từng dòng (frame) của mỗi session.
    """
    def __init__(self, df):
        super(FocusManagementEnv, self).__init__()
        
        # Nhóm dữ liệu theo session_id để tách biệt các video/phiên làm việc
        # Đảm bảo Agent không học "xuyên" từ video này sang video khác
        self.sessions = [group for _, group in df.groupby('session_id')]
        
        # 1. Action Space: 0 (Im lặng), 1 (Soft Nudge), 2 (Hard Nudge)
        self.action_space = spaces.Discrete(3)
        
        # 2. State Space: Số lượng chiều bằng đúng số lượng đặc trưng đã chọn
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, 
            shape=(len(FEATURE_COLS),), 
            dtype=np.float32
        )
        
        self.current_session = None
        self.current_step = 0

    def reset(self, seed=None, options=None):
        """Được gọi mỗi khi bắt đầu một Episode (Một Session/Video mới)"""
        super().reset(seed=seed)
        
        # Chọn ngẫu nhiên một session để đa dạng hóa quá trình huấn luyện
        self.current_session = self.sessions[np.random.randint(len(self.sessions))]
        self.current_step = 0
        
        return self._get_observation(), {}

    def _get_observation(self):
        """Lấy vector trạng thái tại bước hiện tại"""
        row = self.current_session.iloc[self.current_step]
        # Lấy các cột đặc trưng và ép kiểu về float32 (Yêu cầu của Pytorch/Stable-Baselines)
        obs = row[FEATURE_COLS].values.astype(np.float32)
        return obs

    def step(self, action):
        """Thực thi hành động và trả về Reward + State mới"""
        row = self.current_session.iloc[self.current_step]
        
        is_distracted = row['is_distracted_label']
        consecutive_frames = row['consecutive_frames']
        
        reward = 0
        
        # ==========================================
        # LOGIC TÍNH TOÁN PHẦN THƯỞNG (REWARD SHAPING)
        # ==========================================
        if is_distracted == 0:  
            # TRƯỜNG HỢP 1: NGƯỜI DÙNG ĐANG TẬP TRUNG TỐT
            if action == 0:
                reward = 1.0    # Tốt! Im lặng là vàng.
            else:
                reward = -5.0   # Tồi! Làm phiền lúc người ta đang tập trung.
                
        else:                   
            # TRƯỜNG HỢP 2: NGƯỜI DÙNG ĐANG XAO NHÃNG
            # (Giả sử video quay ở tốc độ 3 FPS, 30 frames = 10 giây)
            if consecutive_frames < 30: 
                # Nháy mắt, vươn vai, ngoảnh đi một chút
                if action == 0:
                    reward = 0.5  # Nên giữ im lặng, khoan dung
                elif action == 1:
                    reward = -1.0 # Nhắc hơi sớm
                elif action == 2:
                    reward = -3.0 # Overreact (Quá nhạy cảm)
                    
            elif 30 <= consecutive_frames < 90:
                # Bắt đầu xao nhãng rõ rệt (10s - 30s)
                if action == 0:
                    reward = -2.0 # Bỏ lỡ cơ hội nhắc nhở
                elif action == 1:
                    reward = 3.0  # Tốt! Soft Nudge hợp lý.
                elif action == 2:
                    reward = 0.0  # Hard Nudge có hiệu quả nhưng hơi mạnh tay
                    
            else:
                # Xao nhãng cực kỳ nghiêm trọng (> 30s)
                if action == 0:
                    reward = -4.0 # Phạt nặng vì bỏ mặc người dùng
                elif action == 1:
                    reward = 1.0  # Soft Nudge không đủ đô nữa
                elif action == 2:
                    reward = 5.0  # Hoàn hảo! Khóa màn hình để cảnh tỉnh.

        self.current_step += 1
        
        # Kiểm tra xem đã đọc hết dòng trong session hiện tại chưa
        terminated = self.current_step >= len(self.current_session) - 1
        truncated = False
        
        info = {
            "action": action,
            "reward": reward,
            "consecutive_frames": consecutive_frames
        }
        
        return self._get_observation(), reward, terminated, truncated, info

def prepare_data(csv_path):
    """Đọc và chuẩn hóa dữ liệu từ CSV"""
    print("[*] Đang tải dữ liệu...")
    df = pd.read_csv(csv_path)
    
    # Chuẩn hóa các tính năng để tăng tốc độ hội tụ cho mạng Neural Network
    # StandardScaler đưa mean về 0 và std về 1
    scaler = StandardScaler()
    df[FEATURE_COLS] = scaler.fit_transform(df[FEATURE_COLS])
    
    # Điền các giá trị NaN (nếu có) bằng 0
    df.fillna(0, inplace=True)
    return df, scaler

def run_training_pipeline():
    if not os.path.exists(CSV_FILE):
        print(f"[!] Lỗi: Không tìm thấy file {CSV_FILE}. Hãy chắc chắn bạn đã tạo dữ liệu trước.")
        # Tạo dữ liệu giả (Dummy Data) nếu chưa có file CSV để code không bị lỗi
        print("[*] Đang tạo dữ liệu giả lập (Dummy Data) để chạy thử...")
        dummy_data = pd.DataFrame({
            'session_id': np.repeat([1, 2, 3], 100),
            'timestamp': np.tile(np.arange(100), 3),
            'head_pitch': np.random.normal(0, 15, 300),
            'head_yaw': np.random.normal(0, 20, 300),
            'head_roll': np.random.normal(0, 5, 300),
            'ear_score': np.random.uniform(0.1, 0.4, 300),
            'mar_score': np.random.uniform(0, 0.5, 300),
            'brow_dist': np.random.uniform(20, 50, 300),
            'person_detected': np.random.choice([0, 1], p=[0.1, 0.9], size=300),
            'phone_count': np.random.choice([0, 1], p=[0.8, 0.2], size=300),
            'consecutive_frames': np.tile(np.arange(100), 3), # Tăng dần
            'is_distracted_label': np.random.choice([0, 1], size=300)
        })
        dummy_data.to_csv(CSV_FILE, index=False)

    # 1. Chuẩn bị dữ liệu
    df, scaler = prepare_data(CSV_FILE)
    
    # 2. Khởi tạo môi trường
    env = FocusManagementEnv(df)
    
    # 3. Định nghĩa kiến trúc Agent (DQN)
    # net_arch=[128, 128]: Mạng Neural có 2 lớp ẩn, mỗi lớp 128 nơ-ron
    policy_kwargs = dict(net_arch=[128, 128])
    
    print("\n[*] Khởi tạo Agent DQN...")
    model = DQN(
        "MlpPolicy", 
        env, 
        learning_rate=1e-3, 
        buffer_size=10000,     # Kích thước bộ nhớ kinh nghiệm
        learning_starts=1000,  # Số bước random trước khi bắt đầu học
        batch_size=64, 
        target_update_interval=200, 
        policy_kwargs=policy_kwargs,
        verbose=1              # Hiện log chi tiết trong terminal
    )
    
    # 4. Huấn luyện Agent
    print("[*] Bắt đầu huấn luyện (Training)...")
    # Thay đổi total_timesteps lên cao hơn (VD: 50,000) khi huấn luyện thật
    model.learn(total_timesteps=10000, progress_bar=True)
    
    # 5. Lưu mô hình và Scaler
    model.save("dqn_focus_agent")
    print("[+] Đã lưu mô hình tại: dqn_focus_agent.zip")
    print("[+] Hoàn tất Pipeline Huấn Luyện!")

if __name__ == "__main__":
    run_training_pipeline()

[*] Đang tải dữ liệu...

[*] Khởi tạo Agent DQN...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
[*] Bắt đầu huấn luyện (Training)...
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 99       |
|    ep_rew_mean      | -207     |
|    exploration_rate | 0.624    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1018     |
|    time_elapsed     | 0        |
|    total_timesteps  | 396      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 99       |
|    ep_rew_mean      | -205     |
|    exploration_rate | 0.248    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 978      |
|    time_elapsed     | 0        |
|    total_timesteps  | 792      |
----------------------------------
----------------------------------
| rollout/   

[+] Đã lưu mô hình tại: dqn_focus_agent.zip
[+] Hoàn tất Pipeline Huấn Luyện!
